# BraTS2020 — 3D U-Net baseline (bản tinh gọn)

**Mục đích:** hướng dẫn tái lập pipeline từ phân chia dữ liệu cố định đến huấn luyện, validation, test và trực quan hóa. Đây là bản **đã biên tập từ notebook nghiên cứu**, không phải log thực thi gốc. Bản gốc được giữ riêng để đối chiếu.

| Cấu hình | Giá trị |
|---|---|
| Modalities | T1, T1ce, T2, FLAIR |
| 3D U-Net | MONAI, 4 channels → 4 logits |
| Split | 257 train / 55 internal validation / 56 held-out test |
| Patch | 96³, positive-center probability 0.5 |
| Optimizer / loss | AdamW (1e-4) / DiceCELoss |
| Training | 10 epochs, batch 1, FP16 AMP; validation mỗi 5 epochs |

**Kết quả đã ghi nhận trong lần chạy gốc:** Epoch 10, validation Dice WT/TC/ET = 0.8047 / 0.6658 / 0.6099, mean = **0.6934**; test Dice WT/TC/ET = 0.7964 / 0.7036 / 0.6363, mean = **0.7121**. Các giá trị này là dữ liệu lịch sử, **không phải kết quả tự động sinh ra khi chạy lại notebook đã tinh gọn**.

**Chuẩn bị trước khi chạy:** bật Tesla T4, attach bộ dữ liệu BraTS2020 vào Kaggle; bảo đảm GitHub branch chứa `src/data/`, `src/models/unet3d.py`, `src/evaluation/metrics.py` và split manifest. Nếu chỉ muốn đọc notebook hoặc đưa lên GitHub, **không cần chạy lại**.
**Code organization:** training loop và checkpoint được import từ `src/training/`; validation từ `src/evaluation/validation.py`. Cần copy/commit đủ các module trước khi chạy notebook.


## 1. Environment & imports

In [ ]:
# Cài đặt khi môi trường Kaggle chưa có MONAI.
# Nếu đã có, có thể bỏ qua cell này.
%pip install -q monai

In [ ]:
# 03 BASELINE TRAINING — STEP 1: ENVIRONMENT SETUP

from pathlib import Path
import subprocess
import sys
import torch
import pandas as pd

REPO_URL = (
    "https://github.com/"
    "tthanh81/brats2020-segmentation.git"
)

BRANCH = "refactor/data-pipeline"

PROJECT_ROOT = Path(
    "/kaggle/working/brats2020-segmentation"
)

DATA_ROOT = Path(
    "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation"
)

# 1. Clone repository.
if not PROJECT_ROOT.exists():

    subprocess.run(
        [
            "git", "clone",
            "--branch", BRANCH,
            "--single-branch",
            REPO_URL,
            str(PROJECT_ROOT),
        ],
        check=True,
    )

    print("Repository cloned successfully.")

else:
    assert (PROJECT_ROOT / ".git").is_dir(), (
        "Project directory exists but is not a Git repository."
    )

    print("Existing repository detected.")

# 2. Verify Git branch.
current_branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=PROJECT_ROOT,
    text=True,
).strip()

assert current_branch == BRANCH

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=PROJECT_ROOT,
    text=True,
).strip()

# 3. Configure Python imports.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 4. Verify dataset and source modules.
assert DATA_ROOT.is_dir()

required_files = [
    "splits/split_manifest_seed42.csv",
    "src/data/preprocessing.py",
    "src/data/dataset.py",
    "src/data/patches.py",
    "src/models/unet3d.py",
    "src/evaluation/metrics.py",
]

for filename in required_files:
    assert (PROJECT_ROOT / filename).is_file(), (
        f"Missing file: {filename}"
    )

# 5. Verify GPU.
assert torch.cuda.is_available(), (
    "GPU is unavailable. Enable GPU in Kaggle settings."
)

device = torch.device("cuda")

print("\nPyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# 6. Verify patient split.
MANIFEST_PATH = (
    PROJECT_ROOT / "splits/split_manifest_seed42.csv"
)

split_manifest = pd.read_csv(MANIFEST_PATH)

expected_counts = {
    "train": 257,
    "internal_validation": 55,
    "test": 56,
}

assert split_manifest["subject_id"].is_unique

assert (
    split_manifest["split"].value_counts().to_dict()
    == expected_counts
)

print("\nGit branch:", current_branch)
print("Git commit:", commit)

print("\nTraining subjects: 257")
print("Validation subjects: 55")
print("Test subjects: 56")

print("\nPASS: Baseline training environment ready.")

## 2. Cấu hình và DataLoader

Hai worker được chọn sau benchmark 0 vs 2 workers trong bản gốc. Phép đo benchmark không cần lặp lại mỗi lần chạy. Tập validation và test dùng **full volume**, không lấy patch ngẫu nhiên.

In [ ]:
import os
import time
import json
import zipfile
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader
from monai.inferers import sliding_window_inference

from src.data.dataset import BraTSVolumeDataset
from src.data.patches import RandomPatchDataset
from src.models.unet3d import build_unet3d, build_loss
from src.evaluation.metrics import brats_dice_scores

SEED, NUM_EPOCHS, VAL_INTERVAL = 42, 10, 5
LEARNING_RATE, BATCH_SIZE, NUM_WORKERS = 1e-4, 1, 2
PATCH_SIZE, POS_PROB = (96, 96, 96), 0.5
RUN_NAME = "unet3d_baseline_seed42_v1"
RUN_DIR = PROJECT_ROOT / "outputs" / "runs" / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LATEST_CHECKPOINT = CHECKPOINT_DIR / "latest.pt"
BEST_CHECKPOINT = CHECKPOINT_DIR / "best.pt"
HISTORY_PATH = RUN_DIR / "training_history.csv"

# Các lựa chọn dưới đây mặc định False để không vô tình train/test khi Run All.
RUN_TRAINING = False
RUN_TEST = False
RUN_FIGURES = False
# Nếu muốn tiếp tục training từ file checkpoint, đặt đường dẫn cụ thể bên dưới.
RESUME_CHECKPOINT = None

def get_split_dataset(split):
    frame = split_manifest[split_manifest["split"] == split].reset_index(drop=True).copy()
    return BraTSVolumeDataset(manifest=frame, data_root=DATA_ROOT)

train_dataset = get_split_dataset("train")
val_dataset = get_split_dataset("internal_validation")
test_dataset = get_split_dataset("test")
assert (len(train_dataset), len(val_dataset), len(test_dataset)) == (257, 55, 56)

train_patch_dataset = RandomPatchDataset(
    base_dataset=train_dataset, patch_size=PATCH_SIZE, pos_prob=POS_PROB
)
train_patch_loader = DataLoader(
    train_patch_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=2, drop_last=False,
)
assert len(train_patch_loader) == 257

print("Data ready:", len(train_patch_loader), "train batches,",len(val_dataset), "val,", len(test_dataset), "test")

## 3. Validation trên toàn bộ MRI

Phương pháp: MONAI sliding window 96³, overlap 25%, tổng hợp logits trên CPU. Dice tính theo bệnh nhân, sau đó lấy trung bình (macro average). Khi cả hai mask ET đều rỗng, Dice được quy ước là 1; xem `src/evaluation/metrics.py`.

In [ ]:
# Module used by both internal validation and independent test.
from src.evaluation.validation import validate_model

print("Full-volume validation function imported.")


## 4. Checkpoint và AMP-safe training

Giữ **phiên bản `train_one_epoch()` cuối cùng đã được sử dụng thành công**: chỉ clip nếu gradient hữu hạn; nếu overflow, `GradScaler` bỏ qua optimizer step và giảm scale. Lưu được model, optimizer, scaler và thống kê epoch. `latest.pt` dùng tiếp tục; `best.pt` dùng đánh giá.

**Resume:** nếu một cell training chết giữa epoch, không tiếp tục trên object GradScaler cũ; hãy khởi tạo model, optimizer, GradScaler mới rồi nạp checkpoint. Điều này chỉ khôi phục tới ranh giới epoch, không tái lập chính xác thứ tự patch.

In [ ]:
# Persist model, AdamW and AMP state at completed epoch boundaries.
from src.training.checkpoints import save_checkpoint, load_checkpoint

print("Checkpoint helpers imported.")


In [ ]:
# Same AMP-safe single-epoch function that was used for epochs 4–10.
from src.training.engine import train_one_epoch

print("AMP-safe training function imported.")


In [ ]:
import pandas as pd

def run_training_until(stop_epoch):
    """
    Continue training from next_epoch through stop_epoch.

    Checkpoint policy:
        latest.pt -> after each completed epoch
        best.pt   -> when validation mean Dice improves
    """

    global next_epoch
    global global_step
    global best_val_dice
    global training_history

    assert 1 <= stop_epoch <= NUM_EPOCHS

    if next_epoch > stop_epoch:
        print(
            f"Nothing to train: next_epoch={next_epoch}, "
            f"stop_epoch={stop_epoch}"
        )
        return

    for epoch in range(next_epoch, stop_epoch + 1):

        print("\n" + "=" * 60)
        print(f"STARTING EPOCH {epoch}/{NUM_EPOCHS}")
        print("=" * 60)

        # -----------------------------------------
        # 1. Train one epoch
        # -----------------------------------------

        result = train_one_epoch(
            model=model,
            loader=train_patch_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
            epoch=epoch,
            log_interval=25,
        )

        global_step += result["optimizer_steps"]

        # -----------------------------------------
        # 2. Prepare history row
        # -----------------------------------------

        history_row = {
            "epoch": epoch,
            "train_loss": result["mean_loss"],
            "epoch_time_seconds": result["epoch_time_seconds"],
            "global_step": global_step,
            "positive_patches": result["positive_patches"],
            "tumor_patches": result["tumor_patches"],
            "val_WT": None,
            "val_TC": None,
            "val_ET": None,
            "val_mean_dice": None,
        }

        # -----------------------------------------
        # 3. Save checkpoint immediately after training
        # -----------------------------------------

        save_checkpoint(
            path=LATEST_CHECKPOINT,
            model=model,
            optimizer=optimizer,
            scaler=scaler,
            epoch=epoch,
            global_step=global_step,
            best_val_dice=best_val_dice,
            config=training_config,
        )

        print(f"\nTraining checkpoint saved: epoch {epoch}")

        # -----------------------------------------
        # 4. Full validation at scheduled epochs
        # -----------------------------------------

        if epoch % VAL_INTERVAL == 0:

            print(f"\nRunning full validation at epoch {epoch}")

            val_summary, val_details = validate_model(
                model=model,
                dataset=val_dataset,
                device=device,
                roi_size=PATCH_SIZE,
                overlap=0.25,
            )

            history_row["val_WT"] = val_summary["WT"]
            history_row["val_TC"] = val_summary["TC"]
            history_row["val_ET"] = val_summary["ET"]
            history_row["val_mean_dice"] = (
                val_summary["mean_dice"]
            )

            # Save subject-level validation results.
            val_details.to_csv(
                RUN_DIR / f"validation_epoch_{epoch:02d}.csv",
                index=False,
            )

            print("\nVALIDATION RESULTS")
            print(f"WT: {val_summary['WT']:.4f}")
            print(f"TC: {val_summary['TC']:.4f}")
            print(f"ET: {val_summary['ET']:.4f}")
            print(
                "Mean Dice:",
                f"{val_summary['mean_dice']:.4f}",
            )

            # Update best checkpoint.
            current_dice = val_summary["mean_dice"]

            if (
                best_val_dice is None
                or current_dice > best_val_dice
            ):

                best_val_dice = current_dice

                save_checkpoint(
                    path=BEST_CHECKPOINT,
                    model=model,
                    optimizer=optimizer,
                    scaler=scaler,
                    epoch=epoch,
                    global_step=global_step,
                    best_val_dice=best_val_dice,
                    config=training_config,
                )

                print("New best checkpoint saved.")

            # Refresh latest checkpoint with updated best score.
            save_checkpoint(
                path=LATEST_CHECKPOINT,
                model=model,
                optimizer=optimizer,
                scaler=scaler,
                epoch=epoch,
                global_step=global_step,
                best_val_dice=best_val_dice,
                config=training_config,
            )

        # -----------------------------------------
        # 5. Save history
        # -----------------------------------------

        training_history.append(history_row)

        pd.DataFrame(training_history).to_csv(
            HISTORY_PATH,
            index=False,
        )

        next_epoch = epoch + 1

        # -----------------------------------------
        # 6. Epoch summary
        # -----------------------------------------

        print("\n" + "-" * 60)
        print(f"EPOCH {epoch} COMPLETED")
        print("-" * 60)

        print(
            "Mean training loss:",
            f"{result['mean_loss']:.6f}",
        )

        print(
            "Epoch time:",
            f"{result['epoch_time_seconds']:.2f}s",
        )

        print("Global step:", global_step)
        print("Next epoch:", next_epoch)

        print("Latest checkpoint:", LATEST_CHECKPOINT)

        if best_val_dice is not None:
            print("Best validation Dice:", best_val_dice)

    print("\nPASS: Requested training epochs completed.")

## 5. Khởi tạo hoặc resume, sau đó training (tùy chọn)

Mặc định `RUN_TRAINING=False`. Để **train lại**, chỉ bật khi Kaggle có dataset và bạn muốn chạy 10 epoch từ đầu; dùng `RUN_NAME` mới để không ghi đè. Để **resume**, chỉ định `RESUME_CHECKPOINT`, bảo đảm file `training_history.csv` cùng run vẫn tồn tại. Không chạy lại với cùng tên run nếu đã có `latest.pt` nhưng chưa thiết lập resume.

**Quan trọng:** mô hình huấn luyện 10 epoch của dự án đã hoàn tất. Không cần chạy những cell này chỉ để commit notebook lên GitHub.

In [ ]:
training_config = {
    "model": "MONAI_UNet3D", "seed": SEED,
    "num_epochs": NUM_EPOCHS, "val_interval": VAL_INTERVAL,
    "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE,
    "patch_size": PATCH_SIZE, "pos_prob": POS_PROB,
    "num_workers": NUM_WORKERS, "num_classes": 4,
}

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
model = build_unet3d().to(device)
criterion = build_loss().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
next_epoch, global_step, best_val_dice = 1, 0, None
training_history = []

if RUN_TRAINING:
    if RESUME_CHECKPOINT is not None:
        # Tạo mới GradScaler: tránh trạng thái unscale_ dang dở ở runtime cũ.
        scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
        checkpoint = load_checkpoint(RESUME_CHECKPOINT, model, optimizer, scaler)
        next_epoch = int(checkpoint["epoch"]) + 1
        global_step = int(checkpoint["global_step"])
        best_val_dice = checkpoint["best_val_dice"]
        if not HISTORY_PATH.is_file():
            raise FileNotFoundError("Resume cần training_history.csv cùng run.")
        training_history = pd.read_csv(HISTORY_PATH).to_dict("records")
        if len(training_history) != next_epoch - 1:
            raise ValueError("History không khớp checkpoint; kiểm tra lại run.")
    elif LATEST_CHECKPOINT.exists():
        raise FileExistsError(
            "latest.pt đã tồn tại. Đổi RUN_NAME hoặc đặt RESUME_CHECKPOINT."
        )
print("Model params:", sum(p.numel() for p in model.parameters()))
print("Train enabled:", RUN_TRAINING, "Next epoch:", next_epoch, "Global step:", global_step)

In [ ]:
# Chỉ chạy khi RUN_TRAINING=True; validation ở epochs 5 và 10.
if RUN_TRAINING:
    run_training_until(stop_epoch=NUM_EPOCHS)
else:
    print("SKIP training: RUN_TRAINING=False (checkpoint/result gốc đã lưu riêng).")

## 6. Đánh giá trên tập test giữ lại

`RUN_TEST=True` chỉ sử dụng **best checkpoint đã chọn theo validation**. Không dùng test Dice để lựa chọn lại checkpoint hoặc tinh chỉnh hyperparameter. Trong môi trường Kaggle mới, cần giải nén/copy file `best.pt` đã sao lưu đến `BEST_CHECKPOINT` trước khi bật cờ này.

In [ ]:
if RUN_TEST:
    if not BEST_CHECKPOINT.is_file():
        raise FileNotFoundError(f"Missing best checkpoint: {BEST_CHECKPOINT}")
    best_checkpoint = torch.load(BEST_CHECKPOINT, map_location="cpu", weights_only=True)
    test_model = build_unet3d().to(device)
    test_model.load_state_dict(best_checkpoint["model_state_dict"])
    test_model.eval().requires_grad_(False)
    test_summary, test_results_df = validate_model(
        model=test_model, dataset=test_dataset, device=device,
        roi_size=PATCH_SIZE, overlap=0.25,
    )
    output_dir = RUN_DIR / "test_evaluation"
    output_dir.mkdir(parents=True, exist_ok=True)
    test_results_df.to_csv(output_dir / "test_results_56subjects.csv", index=False)
    et_present = test_results_df["ET_present"]
    summary = {
        "checkpoint_epoch": int(best_checkpoint["epoch"]),
        "num_subjects": int(test_summary["num_subjects"]),
        "WT": test_summary["WT"], "TC": test_summary["TC"],
        "ET": test_summary["ET"], "mean_dice": test_summary["mean_dice"],
        "num_ET_present": int(et_present.sum()),
        "num_ET_absent": int((~et_present).sum()),
        "ET_mean_when_present": float(test_results_df.loc[et_present, "ET"].mean()),
        "total_time_seconds": test_summary["total_time_seconds"],
        "avg_time_seconds": test_summary["avg_time_seconds"],
    }
    (output_dir / "test_summary.json").write_text(json.dumps(summary, indent=2))
    print("FINAL TEST:", {r: round(summary[r], 4) for r in ["WT", "TC", "ET", "mean_dice"]})
else:
    print("SKIP held-out test: RUN_TEST=False")

## 7. Phân tích thống kê & biểu đồ (dựa trên CSV đã lưu)

Không cần suy luận lại. Bật cell này khi đã có `test_results_56subjects.csv`, chẳng hạn sau khi giải nén bản backup. Mean/median/std và các ca ET-absent giúp giải thích hạn chế của mô hình.

In [ ]:
from pathlib import Path
from matplotlib import pyplot as plt

results_path = RUN_DIR / "test_evaluation" / "test_results_56subjects.csv"
analysis_dir = RUN_DIR / "test_evaluation" / "analysis"

if results_path.is_file():
    df = pd.read_csv(results_path)
    assert len(df) == 56
    regions = ["WT", "TC", "ET"]
    print("DICE DISTRIBUTION:\n", df[regions].agg(["mean", "median", "std", "min", "max"]).round(4))
    df["mean_dice"] = df[regions].mean(axis=1)
    print("\nLOWEST 5:\n", df.nsmallest(5, "mean_dice")[["subject_id", *regions, "mean_dice"]])
    et_absent = df.loc[~df["ET_present"].astype(bool), ["subject_id", "ET"]]
    print("\nET-ABSENT:\n", et_absent)
    analysis_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.boxplot([df[r].values for r in regions], tick_labels=regions, showmeans=True)
    ax.set(title="BraTS2020 — Test Dice Distribution", ylabel="Dice", ylim=(0, 1.05))
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(analysis_dir / "test_dice_boxplot.png", dpi=200)
    plt.show()
else:
    print("SKIP analysis: test_results_56subjects.csv chưa có trong RUN_DIR.")

## 8. Trực quan hóa ví dụ & phân tích lỗi (tùy chọn)

Ba trường hợp đã chọn từ **kết quả test hiện có**: `007` (minh họa tốt), `280` (TC kém), `268` (ET false positive). Các Dice báo cáo là của **3D volume**, hình chỉ một lát cắt axial. Không dùng hình test để tiếp tục điều chỉnh mô hình.

Mặc định `RUN_FIGURES=False`. Nếu bật, cần có `best.pt` và dataset test.

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

seg_cmap = ListedColormap(["#000000", "#e45756", "#4c78a8", "#f2cf5b"])
seg_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], seg_cmap.N)
err_cmap = ListedColormap(["#000000", "#3cb371", "#e45756", "#ab63ce"])


def infer_test_subject(subject_id, model):
    frame = split_manifest[split_manifest["split"] == "test"].reset_index(drop=True)
    matches = frame.index[frame["subject_id"] == subject_id].tolist()
    if len(matches) != 1:
        raise KeyError(subject_id)
    sample = test_dataset[matches[0]]
    with torch.inference_mode(), torch.autocast(
        device_type="cuda", dtype=torch.float16, enabled=(device.type == "cuda")
    ):
        logits = sliding_window_inference(
            inputs=sample["image"].unsqueeze(0), roi_size=PATCH_SIZE,
            sw_batch_size=1, predictor=model, overlap=0.25,
            mode="gaussian", sw_device=device, device=torch.device("cpu"),
        )
    prediction = logits.argmax(dim=1).squeeze(0).to(torch.uint8)
    scores = brats_dice_scores(prediction, sample["label"])
    return sample, prediction, scores


def plot_subject(subject_id, model, mode="standard"):
    sample, pred_t, scores = infer_test_subject(subject_id, model)
    flair = sample["image"][3].cpu().numpy()
    gt = sample["label"].cpu().numpy()
    pred = pred_t.cpu().numpy()
    if mode == "et_fp":
        assert not (gt == 3).any(), "ET ground truth must be empty"
        z = int(np.argmax((pred == 3).sum(axis=(0, 1))))
    else:
        z = int(np.argmax((gt > 0).sum(axis=(0, 1))))
    image, gt_slice, pred_slice = [np.rot90(a[:, :, z]) for a in (flair, gt, pred)]
    titles = ["MRI FLAIR", "Ground Truth", "3D U-Net Prediction"]
    if mode == "wt_error": titles.append("WT Error Map")
    if mode == "et_fp": titles.append("False Positive ET")
    fig, axes = plt.subplots(1, len(titles), figsize=(5 * len(titles), 5))
    for ax, title in zip(axes, titles):
        ax.imshow(image, cmap="gray")
        ax.set_title(title)
        ax.axis("off")
    for i, mask in [(1, gt_slice), (2, pred_slice)]:
        axes[i].imshow(np.ma.masked_equal(mask, 0), cmap=seg_cmap,
                       norm=seg_norm, alpha=0.65, interpolation="nearest")
    if mode == "wt_error":
        gt_wt, pred_wt = gt_slice > 0, pred_slice > 0
        errors = np.zeros_like(gt_slice, dtype=np.uint8)
        errors[gt_wt & pred_wt] = 1
        errors[~gt_wt & pred_wt] = 2
        errors[gt_wt & ~pred_wt] = 3
        axes[3].imshow(np.ma.masked_equal(errors, 0), cmap=err_cmap,
                       norm=seg_norm, alpha=0.75, interpolation="nearest")
    if mode == "et_fp":
        axes[3].imshow(np.ma.masked_where(pred_slice != 3, np.ones_like(pred_slice)),
                       cmap=ListedColormap(["#ff3030"]), vmin=0, vmax=1,
                       alpha=0.9, interpolation="nearest")
    fig.suptitle(
        f"{subject_id} | axial z={z} | "
        f"WT={scores['WT']:.4f}, TC={scores['TC']:.4f}, ET={scores['ET']:.4f}"
    )
    fig.tight_layout()
    folder = RUN_DIR / "test_evaluation" / "analysis"
    folder.mkdir(parents=True, exist_ok=True)
    filepath = folder / f"{subject_id}_{mode}.png"
    fig.savefig(filepath, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", filepath)
    if mode == "et_fp":
        print("ET voxels: GT", int((gt == 3).sum()), ", predicted", int((pred == 3).sum()))

if RUN_FIGURES:
    if not BEST_CHECKPOINT.is_file():
        raise FileNotFoundError(BEST_CHECKPOINT)
    if "test_model" not in globals():
        visual_checkpoint = torch.load(BEST_CHECKPOINT, map_location="cpu", weights_only=True)
        test_model = build_unet3d().to(device)
        test_model.load_state_dict(visual_checkpoint["model_state_dict"])
        test_model.eval().requires_grad_(False)
    plot_subject("BraTS20_Training_007", test_model, "standard")
    plot_subject("BraTS20_Training_280", test_model, "wt_error")
    plot_subject("BraTS20_Training_268", test_model, "et_fp")
else:
    print("SKIP figures: RUN_FIGURES=False")

## 9. Kết quả và bàn giao

- Checkpoint đã huấn luyện (`best.pt`, `latest.pt`) và file test được lưu trong **ZIP backup riêng**, **không commit các file `.pt` lên GitHub**.
- Commit notebook gọn này cùng `src/evaluation/metrics.py`, mô hình, data pipeline và manifest (các đường dẫn dữ liệu MRI gốc không nằm trong Git).
- Lần chạy gốc ghi nhận **Mean Validation Dice 0.6934** và **Mean Test Dice 0.7121**; ET-absent: 4/56 bệnh nhân, cả bốn có Dice ET = 0 theo quy ước hiện tại. Hạn chế: dự đoán nhầm nhãn edema/NCR và ET false-positive lẻ tẻ.
- Muốn chạy lại trên môi trường mới: clone đúng branch, attach dataset, cài MONAI, commit `metrics.py`, đặt run riêng, bật `RUN_TRAINING=True`; sau khi chọn best theo validation, bật `RUN_TEST=True`. Không kỳ vọng kết quả số học giống từng chữ số vì pipeline patch ngẫu nhiên/DataLoader chưa bảo đảm tái lập tuyệt đối sau restart.